<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/45_explainable_agent/explainable_agent_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explainable AI Agent

This notebook implements an AI agent that:
- Classifies query intent
- Selects appropriate tools
- Generates reasoning
- Produces structured outputs with confidence

In [ ]:
import re

In [ ]:
def classify_intent(query):
    query = query.lower()

    if any(word in query for word in ["%", "calculate", "sum", "add", "multiply"]):
        return "calculation"
    elif any(word in query for word in ["what is", "who is", "define"]):
        return "knowledge"
    else:
        return "explanation"

In [ ]:
def generate_reason(query, intent):
    if intent == "calculation":
        return "The query involves a mathematical computation."
    elif intent == "knowledge":
        return "The query is asking for factual information."
    else:
        return "The query requires a general explanation."

In [ ]:
def calculator_tool(query):
    match = re.search(r'(\d+)%.*?(\d+)', query)
    if match:
        percent = float(match.group(1))
        number = float(match.group(2))
        result = (percent / 100) * number

        steps = [
            f"Convert {percent}% to decimal → {percent/100}",
            f"Multiply {percent/100} × {number}"
        ]
        return result, steps

    return "Calculation not supported", []


def knowledge_tool(query):
    query = query.lower()

    if "artificial intelligence" in query:
        return (
            "Artificial Intelligence is the simulation of human intelligence in machines that are programmed to think and learn.",
            ["Identified topic: Artificial Intelligence", "Retrieved predefined knowledge"]
        )

    return "No knowledge available.", ["Knowledge base lookup failed"]

def explanation_tool(query):
    query = query.lower()

    if "machine learning" in query:
        return (
            "Machine Learning is a subset of AI where systems learn patterns from data to make predictions or decisions without being explicitly programmed.",
            ["Identified topic: Machine Learning", "Generated simple explanation"]
        )

    return "Explanation not available.", ["Explanation fallback triggered"]

In [ ]:
def compute_confidence(intent, answer):
    if isinstance(answer, str) and "not available" in answer.lower():
        return "Low"

    if intent == "calculation":
        return "High"
    elif intent == "knowledge":
        return "High"
    else:
        return "Medium"

In [ ]:
def explainable_agent(query):
    intent = classify_intent(query)
    reason = generate_reason(query, intent)

    if intent == "calculation":
        answer, steps = calculator_tool(query)
        action = "Calculator Tool"
    elif intent == "knowledge":
        answer, steps = knowledge_tool(query)
        action = "Knowledge Tool"
    else:
        answer, steps = explanation_tool(query)
        action = "Explanation Tool"

    confidence = compute_confidence(intent, answer)

    return {
        "query": query,
        "intent": intent,
        "reason": reason,
        "action": action,
        "steps": steps,
        "final_answer": answer,
        "confidence": confidence
    }

In [13]:
queries = [
    "What is 25% of 200?",
    "What is Artificial Intelligence?",
    "Explain machine learning"
]

for q in queries:
    result = explainable_agent(q)
    print("\n==============================")
    for k, v in result.items():
        print(f"{k.upper()}: {v}")


QUERY: What is 25% of 200?
INTENT: calculation
REASON: The query involves a mathematical computation.
ACTION: Calculator Tool
STEPS: ['Convert 25.0% to decimal → 0.25', 'Multiply 0.25 × 200.0']
FINAL_ANSWER: 50.0
CONFIDENCE: High

QUERY: What is Artificial Intelligence?
INTENT: knowledge
REASON: The query is asking for factual information.
ACTION: Knowledge Tool
STEPS: ['Identified topic: Artificial Intelligence', 'Retrieved predefined knowledge']
FINAL_ANSWER: Artificial Intelligence is the simulation of human intelligence in machines that are programmed to think and learn.
CONFIDENCE: High

QUERY: Explain machine learning
INTENT: explanation
REASON: The query requires a general explanation.
ACTION: Explanation Tool
STEPS: ['Identified topic: Machine Learning', 'Generated simple explanation']
FINAL_ANSWER: Machine Learning is a subset of AI where systems learn patterns from data to make predictions or decisions without being explicitly programmed.
CONFIDENCE: Medium
